In [ ]:
!pip install -q transformers peft bitsandbytes datasets accelerate evaluate rouge_score trl nltk

In [ ]:
import torch
import time
import pandas as pd
import matplotlib.pyplot as plt
import evaluate

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)

from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

In [ ]:
model_id = "unsloth/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

In [ ]:

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
dataset = load_dataset("squad")

train_val = dataset["train"].train_test_split(
    train_size=10000,
    test_size=1000,
    seed=42
)

train_ds = train_val["train"]
val_ds = train_val["test"]
test_ds = dataset["validation"].select(range(1000))

In [ ]:
def map_squad_to_text(example):
    ans = example['answers']['text'][0] if len(example['answers']['text']) > 0 else ""
    return {
        "text": f"Context: {example['context']}\n"
                f"Question: {example['question']}\n"
                f"Answer: {ans}{tokenizer.eos_token}"
    }

train_ds = train_ds.map(map_squad_to_text)
val_ds = val_ds.map(map_squad_to_text)

In [ ]:
# 7. EVALUATION ENGINE

# Load evaluation metrics
rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")


# Format prompt for inference (no answer included)
def format_squad_prompt(example):
    return (
        f"Context: {example['context']}\n"
        f"Question: {example['question']}\n"
        f"Answer: "
    )


# Evaluation function
def run_evaluation(model, dataset, num_samples=50):
    model.eval()
    predictions = []
    references = []

    print(f"Evaluating {num_samples} samples...")

    for i in range(num_samples):
        item = dataset[i]

        prompt = format_squad_prompt(item)
        reference_answer = item['answers']['text'][0]

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=64,
                do_sample=False
            )

        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Extract answer part only
        predicted_answer = generated_text.split("Answer:")[-1].strip()

        predictions.append(predicted_answer)
        references.append(reference_answer)

    # Compute metrics
    rouge_results = rouge_metric.compute(
        predictions=predictions,
        references=references
    )

    bleu_results = bleu_metric.compute(
        predictions=predictions,
        references=[[r] for r in references]
    )

    return rouge_results, bleu_results

In [ ]:
baseline_rouge, baseline_bleu = run_evaluation(base_model, test_ds)

In [ ]:
print("\n--- Baseline Evaluation ---")
print(f"ROUGE-1: {baseline_rouge['rouge1']:.4f}")
print(f"ROUGE-2: {baseline_rouge['rouge2']:.4f}")
print(f"ROUGE-L: {baseline_rouge['rougeL']:.4f}")
print(f"BLEU: {baseline_bleu['bleu']:.4f}")

In [ ]:
sft_config = SFTConfig(
    output_dir="./squad_qlora_results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_steps=150,
    logging_steps=10,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="paged_adamw_8bit",
    report_to="none",
    dataset_text_field="text"
)

# 11. Initialize Trainer
trainer = SFTTrainer(
    model=base_model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=lora_config,
    args=sft_config
)

In [ ]:
trainer.train()

In [ ]:
ft_rouge, ft_bleu = run_evaluation(base_model, test_ds)

In [ ]:
print("\n========== RESULTS ==========")

print("\nBaseline:")
print(f"ROUGE-1: {baseline_rouge['rouge1']:.4f}")
print(f"ROUGE-2: {baseline_rouge['rouge2']:.4f}")
print(f"ROUGE-L: {baseline_rouge['rougeL']:.4f}")
print(f"BLEU: {baseline_bleu['bleu']:.4f}")

print("\nFine-Tuned:")
print(f"ROUGE-1: {ft_rouge['rouge1']:.4f}")
print(f"ROUGE-2: {ft_rouge['rouge2']:.4f}")
print(f"ROUGE-L: {ft_rouge['rougeL']:.4f}")
print(f"BLEU: {ft_bleu['bleu']:.4f}")

In [ ]:
def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Trainable parameters before pruning:", count_trainable_params(base_model))

In [ ]:
ft_rouge, ft_bleu = run_evaluation(base_model, test_ds)

In [ ]:
import torch.nn.utils.prune as prune

In [ ]:
def prune_lora_layers(model, amount=0.5):
    total_pruned = 0

    for name, module in model.named_modules():
        if "lora" in name and hasattr(module, "weight"):

            prune.l1_unstructured(
                module,
                name="weight",
                amount=amount
            )

            prune.remove(module, "weight")
            total_pruned += 1

    print(f"Pruned {total_pruned} LoRA layers")
    return model

In [ ]:
base_model = prune_lora_layers(base_model, amount=0.5)

print("50% pruning complete")

In [ ]:
pruned_rouge, pruned_bleu = run_evaluation(base_model, test_ds)

print("\n--- Pruned QLoRA Evaluation ---")
print(f"ROUGE-1: {pruned_rouge['rouge1']:.4f}")
print(f"ROUGE-2: {pruned_rouge['rouge2']:.4f}")
print(f"ROUGE-L: {pruned_rouge['rougeL']:.4f}")
print(f"BLEU: {pruned_bleu['bleu']:.4f}")

In [ ]:
print("\n========== FINAL COMPARISON ==========")

print("\nBaseline:")
print(f"ROUGE-1: {baseline_rouge['rouge1']:.4f}")

print("\nQLoRA:")
print(f"ROUGE-1: {ft_rouge['rouge1']:.4f}")

print("\nQLoRA + Pruning:")
print(f"ROUGE-1: {pruned_rouge['rouge1']:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models = ["Baseline", "QLoRA", "QLoRA + Pruning"]

rouge1 = [baseline_rouge["rouge1"],
          ft_rouge["rouge1"],
          pruned_rouge["rouge1"]]

rouge2 = [baseline_rouge["rouge2"],
          ft_rouge["rouge2"],
          pruned_rouge["rouge2"]]

rougeL = [baseline_rouge["rougeL"],
          ft_rouge["rougeL"],
          pruned_rouge["rougeL"]]

x = np.arange(len(models))
width = 0.25

plt.figure()
plt.bar(x - width, rouge1, width, label="ROUGE-1")
plt.bar(x, rouge2, width, label="ROUGE-2")
plt.bar(x + width, rougeL, width, label="ROUGE-L")

plt.xticks(x, models)
plt.ylabel("Score")
plt.title("ROUGE Score Comparison")
plt.legend()
plt.show()

In [ ]:
bleu_scores = [baseline_bleu["bleu"],
               ft_bleu["bleu"],
               pruned_bleu["bleu"]]

plt.figure()
plt.bar(models, bleu_scores)
plt.ylabel("BLEU Score")
plt.title("BLEU Score Comparison")
plt.show()

In [ ]:
losses = [log["loss"] for log in trainer.state.log_history if "loss" in log]

plt.figure()
plt.plot(losses)
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("QLoRA Training Loss Curve")
plt.show()

In [ ]:
pruning_levels = [0.0, 0.5]
rouge1_scores = [
    ft_rouge["rouge1"],
    pruned_rouge["rouge1"]
]

plt.figure()
plt.plot(pruning_levels, rouge1_scores, marker="o")
plt.xlabel("Pruning Ratio")
plt.ylabel("ROUGE-1")
plt.title("Performance vs Pruning Level")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

labels = ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU"]

baseline_vals = [baseline_rouge["rouge1"],
                 baseline_rouge["rouge2"],
                 baseline_rouge["rougeL"],
                 baseline_bleu["bleu"]]

qlora_vals = [ft_rouge["rouge1"],
              ft_rouge["rouge2"],
              ft_rouge["rougeL"],
              ft_bleu["bleu"]]

pruned_vals = [pruned_rouge["rouge1"],
               pruned_rouge["rouge2"],
               pruned_rouge["rougeL"],
               pruned_bleu["bleu"]]

# Radar setup
angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()

baseline_vals += baseline_vals[:1]
qlora_vals += qlora_vals[:1]
pruned_vals += pruned_vals[:1]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(6,6), subplot_kw=dict(polar=True))

ax.plot(angles, baseline_vals, label="Baseline")
ax.plot(angles, qlora_vals, label="QLoRA")
ax.plot(angles, pruned_vals, label="Pruned")

ax.fill(angles, baseline_vals, alpha=0.1)
ax.fill(angles, qlora_vals, alpha=0.1)
ax.fill(angles, pruned_vals, alpha=0.1)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels)
ax.set_title("Model Performance Radar Chart")

plt.legend()
plt.show()

In [ ]:
improvement = [
    (ft_rouge["rouge1"] - baseline_rouge["rouge1"]) / baseline_rouge["rouge1"] * 100,
    (ft_rouge["rouge2"] - baseline_rouge["rouge2"]) / baseline_rouge["rouge2"] * 100,
    (ft_rouge["rougeL"] - baseline_rouge["rougeL"]) / baseline_rouge["rougeL"] * 100,
]

metrics = ["ROUGE-1", "ROUGE-2", "ROUGE-L"]

plt.figure()
plt.bar(metrics, improvement)
plt.ylabel("Percentage Improvement (%)")
plt.title("QLoRA Improvement Over Baseline")
plt.show()

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "Baseline": baseline_vals[:-1],
    "QLoRA": qlora_vals[:-1],
    "Pruned": pruned_vals[:-1]
}, index=labels)

df.plot(kind="bar")
plt.title("Normalized Metric Comparison")
plt.ylabel("Score")
plt.show()

In [ ]:
efficiency = ["Baseline", "QLoRA", "Pruned"]
performance = [baseline_rouge["rouge1"],
               ft_rouge["rouge1"],
               pruned_rouge["rouge1"]]

plt.figure()
plt.scatter(efficiency, performance)
plt.title("Efficiency vs Performance")
plt.ylabel("ROUGE-1")
plt.show()

In [ ]:
def get_sparsity(model):
    total = 0
    zeros = 0
    for name, param in model.named_parameters():
        if "lora" in name:
            total += param.numel()
            zeros += (param == 0).sum().item()
    return 100 * zeros / total if total > 0 else 0

sparsity = get_sparsity(base_model)

plt.figure()
plt.bar(["Pruned Model"], [sparsity])
plt.ylabel("Sparsity (%)")
plt.title("LoRA Sparsity After Pruning")
plt.show()